In [ ]:
# AI Travel Agent with Gemini API - Google Colab Setup

# 1. Install required dependencies
!pip install python-dotenv langchain-core langchain-google-genai langgraph sendgrid serpapi streamlit gradio google-generativeai

# 2. Import necessary libraries
import datetime
import operator
import os
import uuid
import json
from typing import Annotated, TypedDict, Optional, List, Dict, Any

from dotenv import load_dotenv
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail
import serpapi
import google.generativeai as genai
from pydantic import BaseModel, Field
from langchain_core.tools import tool
import gradio as gr

# 3. Set up environment variables
# Replace with your actual API keys
os.environ["GOOGLE_API_KEY"] = "AIzaS"  # Get from https://makersuite.google.com/app/apikey
os.environ["FROM_EMAIL"] = "aman24012@iiitd.ac.in"
os.environ["TO_EMAIL"] = "amankumarsg001@gmail.com"
os.environ["EMAIL_SUBJECT"] = "Travel Information"
os.environ["SERPAPI_API_KEY"] = "a07656fa0829de57707a22b19500"  # Get from https://serpapi.com/
os.environ["SENDGRID_API_KEY"] = "SG.uV0OlPjnTyesGXSkI"  # Get from https://sendgrid.com/

# Load environment variables
load_dotenv()

# Configure Gemini
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

CURRENT_YEAR = datetime.datetime.now().year

# Define the agent state
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# System prompt for the travel agent
TOOLS_SYSTEM_PROMPT = f"""You are a smart travel agency assistant. Use the available tools to look up travel information.
You are allowed to make multiple calls (either together or in sequence).
Only look up information when you are sure of what you want.
The current year is {CURRENT_YEAR}. If you don't get flights, try searching until you find them!
If you need to look up information before asking a follow-up question, you are allowed to do that!

In your output, include:
- Links to hotel websites and flight websites (if available)
- Hotel logos and airline logos (if available)
- Always include both flight prices and hotel prices with currency
- Format the response in a clear, readable manner

For hotels, include:
- Hotel name and rating
- Price per night with currency
- Location and amenities
- Booking link if available

For flights, include:
- Airline name
- Flight duration and stops
- Price with currency
- Departure and arrival times
- Booking link if available
"""

# ----- Flight Search Tool -----
class FlightsInput(BaseModel):
    departure_airport: Optional[str] = Field(description='Departure airport code (IATA)')
    arrival_airport: Optional[str] = Field(description='Arrival airport code (IATA)')
    outbound_date: Optional[str] = Field(description='Outbound date (YYYY-MM-DD)')
    return_date: Optional[str] = Field(description='Return date (YYYY-MM-DD)')
    adults: Optional[int] = Field(1, description='Number of adults (default 1)')
    children: Optional[int] = Field(0, description='Number of children (default 0)')
    infants_in_seat: Optional[int] = Field(0, description='Number of infants in seat (default 0)')
    infants_on_lap: Optional[int] = Field(0, description='Number of infants on lap (default 0)')

@tool
def flights_finder(
    departure_airport: str,
    arrival_airport: str,
    outbound_date: str,
    return_date: str = None,
    adults: int = 1,
    children: int = 0,
    infants_in_seat: int = 0,
    infants_on_lap: int = 0
):
    """
    Find flights using the Google Flights engine (via SerpAPI).

    Args:
        departure_airport: Departure airport code (IATA)
        arrival_airport: Arrival airport code (IATA)
        outbound_date: Outbound date (YYYY-MM-DD)
        return_date: Return date (YYYY-MM-DD), optional
        adults: Number of adults (default 1)
        children: Number of children (default 0)
        infants_in_seat: Number of infants in seat (default 0)
        infants_on_lap: Number of infants on lap (default 0)

    Returns:
        Flight search results or error message
    """
    query = {
        'api_key': os.environ.get('SERPAPI_API_KEY'),
        'engine': 'google_flights',
        'hl': 'en',
        'gl': 'us',
        'departure_id': departure_airport,
        'arrival_id': arrival_airport,
        'outbound_date': outbound_date,
        'currency': 'USD',
        'adults': adults,
        'infants_in_seat': infants_in_seat,
        'stops': '1',
        'infants_on_lap': infants_on_lap,
        'children': children
    }

    if return_date:
        query['return_date'] = return_date

    try:
        search = serpapi.search(query)
        result = search.data

        # Return best flights if available
        if 'best_flights' in result:
            return result['best_flights']
        elif 'other_flights' in result:
            return result['other_flights'][:5]  # Return top 5 other flights
        else:
            return "No flights found for the given criteria."

    except Exception as e:
        return f"Error searching flights: {str(e)}"

# ----- Hotel Search Tool -----
@tool
def hotels_finder(
    location: str,
    check_in_date: str,
    check_out_date: str,
    adults: int = 1,
    children: int = 0,
    rooms: int = 1,
    sort_by: int = 8,
    hotel_class: str = None
):
    """
    Find hotels using the Google Hotels engine (via SerpAPI).

    Args:
        location: Location for hotels (e.g., "New York")
        check_in_date: Check-in date (YYYY-MM-DD)
        check_out_date: Check-out date (YYYY-MM-DD)
        adults: Number of adults (default 1)
        children: Number of children (default 0)
        rooms: Number of rooms (default 1)
        sort_by: Sorting parameter (default=8 for rating)
        hotel_class: Filter by hotel class (e.g., "3" or "4")

    Returns:
        List of hotel properties or error message
    """
    query = {
        'api_key': os.environ.get('SERPAPI_API_KEY'),
        'engine': 'google_hotels',
        'hl': 'en',
        'gl': 'us',
        'q': location,
        'check_in_date': check_in_date,
        'check_out_date': check_out_date,
        'currency': 'USD',
        'adults': adults,
        'children': children,
        'rooms': rooms,
        'sort_by': sort_by
    }

    if hotel_class:
        query['hotel_class'] = hotel_class

    try:
        search = serpapi.search(query)
        data = search.data

        if 'properties' in data:
            return data['properties'][:5]  # Return top 5 hotels
        else:
            return "No hotels found for the given criteria."

    except Exception as e:
        return f"Error searching hotels: {str(e)}"

# Available tools
TOOLS = [flights_finder, hotels_finder]

# ----- Agent Class -----
class GeminiTravelAgent:
    def __init__(self):
        # Initialize Gemini model with tools
        self.model = ChatGoogleGenerativeAI(
            model="gemini-pro",
            google_api_key=os.environ["GOOGLE_API_KEY"],
            temperature=0.3,
            convert_system_message_to_human=True
        )

        # Bind tools to the model
        self.tools_model = self.model.bind_tools(TOOLS)

        # Map tool names to functions
        self._tools = {t.name: t for t in TOOLS}

        # Build the state graph
        builder = StateGraph(AgentState)
        builder.add_node('call_tools_llm', self.call_tools_llm)
        builder.add_node('invoke_tools', self.invoke_tools)
        builder.add_node('email_sender', self.email_sender)
        builder.set_entry_point('call_tools_llm')

        # Define edges
        builder.add_conditional_edges(
            'call_tools_llm',
            self.should_continue,
            {'continue': 'invoke_tools', 'end': 'email_sender'}
        )
        builder.add_edge('invoke_tools', 'call_tools_llm')
        builder.add_edge('email_sender', END)

        # Compile the graph
        memory = MemorySaver()
        self.graph = builder.compile(checkpointer=memory, interrupt_before=['email_sender'])

    def should_continue(self, state: AgentState):
        """Determine if we should continue with tool calls or end"""
        last_message = state['messages'][-1]

        # Check if the last message has tool calls
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            return 'continue'
        else:
            return 'end'

    def call_tools_llm(self, state: AgentState):
        """Call the LLM with tools"""
        messages = state['messages']

        # Add system message
        system_message = SystemMessage(content=TOOLS_SYSTEM_PROMPT)
        messages_with_system = [system_message] + messages

        # Invoke the model
        response = self.tools_model.invoke(messages_with_system)

        return {'messages': [response]}

    def invoke_tools(self, state: AgentState):
        """Execute the requested tools"""
        last_message = state['messages'][-1]
        tool_calls = last_message.tool_calls

        tool_messages = []
        for tool_call in tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']

            if tool_name in self._tools:
                try:
                    result = self._tools[tool_name].invoke(tool_args)
                    tool_messages.append(
                        ToolMessage(
                            tool_call_id=tool_id,
                            name=tool_name,
                            content=str(result)
                        )
                    )
                except Exception as e:
                    tool_messages.append(
                        ToolMessage(
                            tool_call_id=tool_id,
                            name=tool_name,
                            content=f"Error: {str(e)}"
                        )
                    )
            else:
                tool_messages.append(
                    ToolMessage(
                        tool_call_id=tool_id,
                        name=tool_name,
                        content=f"Error: Tool {tool_name} not found"
                    )
                )

        return {'messages': tool_messages}

    def email_sender(self, state: AgentState):
        """Email sender node (not used in Gradio version)"""
        return {'messages': []}

# ----- Email Functionality -----
def send_html_email(travel_html: str, sender: str, receiver: str, subject: str) -> str:
    """
    Send email using SendGrid
    """
    message = Mail(
        from_email=sender,
        to_emails=receiver,
        subject=subject,
        html_content=travel_html
    )

    try:
        sg = SendGridAPIClient(os.environ.get('SENDGRID_API_KEY'))
        response = sg.send(message)
        return f"Email sent successfully! Status: {response.status_code}"
    except Exception as e:
        return f"Error sending email: {str(e)}"

# ----- Initialize Agent -----
agent = GeminiTravelAgent()

# ----- Gradio Functions -----
def process_travel_query(user_query: str) -> str:
    """Process travel query and return results"""
    if not user_query.strip():
        return "Please enter a travel query."

    try:
        thread_id = str(uuid.uuid4())
        messages = [HumanMessage(content=user_query)]
        config = {'configurable': {'thread_id': thread_id}}

        # Run the agent
        result = agent.graph.invoke({'messages': messages}, config=config)

        # Get the final response
        final_message = result['messages'][-1]
        return final_message.content

    except Exception as e:
        return f"Error processing query: {str(e)}"

def send_travel_email(travel_info: str, sender: str, receiver: str, subject: str) -> str:
    """Send travel information via email"""
    if not all([travel_info, sender, receiver, subject]):
        return "Error: All fields are required."

    return send_html_email(travel_info, sender, receiver, subject)

# ----- Gradio Interface -----
def create_gradio_interface():
    """Create and return the Gradio interface"""

    with gr.Blocks(title="AI Travel Agent with Gemini") as demo:
        gr.Markdown("# ✈️🌍 AI Travel Agent (Powered by Google Gemini)")
        gr.Markdown("Enter your travel query below and get personalized flight and hotel recommendations!")

        with gr.Row():
            with gr.Column():
                query_input = gr.Textbox(
                    lines=4,
                    placeholder="Example: Find flights from New York to London on June 15, 2024, returning June 22, 2024, and 4-star hotels in London",
                    label="Travel Query"
                )
                query_button = gr.Button("🔍 Search Travel Options", variant="primary")

        travel_output = gr.Markdown("", label="Travel Results")

        # Connect the search functionality
        query_button.click(
            fn=process_travel_query,
            inputs=query_input,
            outputs=travel_output
        )

        gr.Markdown("---")
        gr.Markdown("## 📧 Email Travel Information")
        gr.Markdown("Send the travel information above to your email")

        with gr.Row():
            with gr.Column():
                sender_input = gr.Textbox(
                    label="From Email",
                    placeholder="your-email@example.com"
                )
                receiver_input = gr.Textbox(
                    label="To Email",
                    placeholder="recipient@example.com"
                )
                subject_input = gr.Textbox(
                    label="Subject",
                    value="Your Travel Information"
                )
                email_button = gr.Button("📧 Send Email", variant="secondary")

        email_status = gr.Textbox(label="Email Status", interactive=False)

        # Connect the email functionality
        email_button.click(
            fn=send_travel_email,
            inputs=[travel_output, sender_input, receiver_input, subject_input],
            outputs=email_status
        )

        gr.Markdown("---")
        gr.Markdown("### 📝 Instructions:")
        gr.Markdown("""
        1. **Enter your travel query** - Be specific about dates, locations, and preferences
        2. **Click Search** - The AI will find flights and hotels for you
        3. **Review results** - Check the recommendations and pricing
        4. **Email results** - Optionally send the information to your email

        **Example queries:**
        - "Find flights from NYC to Paris on July 1st, returning July 10th, and luxury hotels in Paris"
        - "Round trip flights from LAX to Tokyo in December 2024, plus budget hotels"
        - "One-way flight from London to Dubai on March 15th and 5-star hotels in Dubai"
        """)

    return demo

# ----- Launch the Application -----
if __name__ == "__main__":
    # Create and launch the Gradio interface
    demo = create_gradio_interface()
    demo.launch(
        server_name="0.0.0.0",
        share=True,
        debug=True
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-generativeai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/usr/local/lib/python3.11/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
